<a href="https://colab.research.google.com/github/Aniketh78/Generative-AI-Lab_Experiments/blob/main/GenAiExp08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [10]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import evaluate
import torch

dataset = load_dataset("banking77")

# Get the ClassLabel feature for the 'label' column to convert int to string
label_feature = dataset['train'].features['label']

def convert_to_qa(example):
    # Convert the integer label to its text representation
    label_text = label_feature.int2str(example["label"])
    return {
        "input_text": "Question: " + example["text"],
        "target_text": label_text
    }

dataset = dataset.map(convert_to_qa)

train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(100))

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

def tokenize(example):
    inputs = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )
    outputs = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )
    inputs["labels"] = outputs["input_ids"]
    return inputs

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=50,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data
)

trainer.train()


model.save_pretrained("banking_model")
tokenizer.save_pretrained("banking_model")


bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")


predictions = []
references = []

for example in test_data:
    input_ids = example["input_ids"].unsqueeze(0)

    output = model.generate(input_ids, max_length=32)
    pred = tokenizer.decode(output[0], skip_special_tokens=True)

    ref = tokenizer.decode(example["labels"], skip_special_tokens=True)

    predictions.append(pred)
    references.append([ref])

# ==============================
# 11. Compute BLEU (AVERAGED)
# ==============================
bleu_result = bleu.compute(predictions=predictions, references=references)

bleu_1 = bleu_result["precisions"][0]
bleu_2 = bleu_result["precisions"][1]
bleu_3 = bleu_result["precisions"][2]
bleu_4 = bleu_result["precisions"][3]

# ==============================
# 12. Compute ROUGE (AVERAGED)
# ==============================
rouge_result = rouge.compute(
    predictions=predictions,
    references=[r[0] for r in references]
)

rouge_1 = rouge_result["rouge1"]
rouge_2 = rouge_result["rouge2"]
rouge_l = rouge_result["rougeL"]

# ==============================
# 13. Final Output (REQUIRED FORMAT)
# ==============================
print("\n=== FINAL EVALUATION SCORES ===")

print(f"BLEU-1  : {bleu_1:.4f}")
print(f"BLEU-2  : {bleu_2:.4f}")
print(f"BLEU-3  : {bleu_3:.4f}")
print(f"BLEU-4  : {bleu_4:.4f}")

print(f"ROUGE-1 : {rouge_1:.4f}")
print(f"ROUGE-2 : {rouge_2:.4f}")
print(f"ROUGE-L : {rouge_l:.4f}")

# ==============================
# 14. Inference Function
# ==============================
def ask(query):
    input_text = "Question: " + query
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    output = model.generate(input_ids, max_length=32)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# ==============================
# 15. Test Query
# ==============================
print("\n=== SAMPLE TEST ===")
query = "I lost my card, what should I do?"
print("Q:", query)
print("A:", ask(query))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss
50,4.838174
100,1.590953
150,1.073089
200,0.765020


Step,Training Loss
50,4.838174
100,1.590953
150,1.073089
200,0.765020
250,0.663871
300,0.593061
350,0.559364
400,0.515011
450,0.514322
500,0.487582


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== FINAL EVALUATION SCORES ===
BLEU-1  : 0.4467
BLEU-2  : 0.2379
BLEU-3  : 0.1205
BLEU-4  : 0.0848
ROUGE-1 : 0.3809
ROUGE-2 : 0.1193
ROUGE-L : 0.3614

=== SAMPLE TEST ===
Q: I lost my card, what should I do?
A: lost_card_deposit


In [7]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=e67781b4a1ef98f201bc7e15abd8a64bc626161159b4cfd02e2de6326cef1ff5
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
